# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["LOG_LEVEL"] = "INFO"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: mps


In [4]:
from mllm_shap.connectors import LiquidAudio, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter
from mllm_shap.shap import Explainer, McShapExplainer
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import PowerShiftNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df

# Usage

Define a `LiquidAudio` model (this call loads it into memory!).

Create a compact explainer using `McShapExplainer` with `num_samples=-1` (iterate over the entire mask space — equivalent to Precise enumeration via sampling) and contextual embeddings. `PowerShiftNormalizer(power=2.0)` shifts values so the minimum is 0, raises them to the power of 2, then normalizes so they sum to 1.

In [5]:
model = LiquidAudio(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history
shap = McShapExplainer(
    num_samples=-1,
    mode=Mode.CONTEXTUAL,  # use contextual embeddings, default
    # use mean pooling to reduce token embeddings to single embedding per audio, default
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=PowerShiftNormalizer(
        power=2.0
    ),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

W0505 13:31:48.204000 1899 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Create new chat that treats Assistant messages as system, that is will ignore them for shapley values calculation - they will be feed to each prompt as system messages. This significantly reduces number of requests needed for multi turn expandability, yet might not be possible due to business requirements. 

To further reduce number of calls we exclude punctuation tokens.

In [6]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.SYSTEM_ASSISTANT,
    token_filter=ExcludePunctuationTokensFilter(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.SYSTEM)
chat.add_text("You are a helpful assistant that answers questions briefly.")
chat.end_turn()

chat.new_turn(Role.USER)
chat.add_text("Who are you?")
chat.end_turn()

Let's have a look at chat representation:

In [7]:
chat.get_conversation()

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=None)],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Let's calculate shapley values for current conversation.

Verbose=True allows us to access history object (descried later). Generation kwargs allows to customize model interference - here we limit it to 64 tokens and change text_temperature from default 0.0 to 0.2. 

In [8]:
generation_kwargs = {
    "max_new_tokens": 64,
    "model_config": ModelConfig(text_temperature=0.2),
}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,  # show progress bar during generation, default
)

2026-05-05 13:31:53,344 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2026-05-05 13:31:57,013 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 3 (up to 7 additional calls)


Monte Carlo SHAP:   0%|          | 0/4 [00:00<?, ?it/s]

2026-05-05 13:32:00,500 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=4 cache_hits=0 cache_misses=4 skipped_filtered=0 model_elapsed_ms=2700.70
2026-05-05 13:32:00,500 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=5 yielded=4 skipped(full_or_empty)=1 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=3067.72


`result` has the following fields:

- `full_chat` — chat with the base response (generated from the full, unmasked input) and with SHAP values attached
- `source_chat` — the original chat fed to the explainer
- `history` — list of all (mask, mask_hash, masked_chat, model_response) tuples evaluated during explanation

The cache stored in `full_chat` holds SHAP values, embeddings, and masks. On subsequent calls it is reused to skip already-evaluated masks.

With `ExcludePunctuationTokensFilter` and query `"Who are you?"`, 3 tokens are explainable (`Who`, `are`, `you`). `num_samples=-1` exhausts the full mask space: 2³ − 2 = 6 non-trivial subsets. Each history entry is a tuple of:

- mask (1D bool tensor)
- mask hash
- masked chat (`None` if response came from cache)
- model response

Let's see all masked prompts evaluated:

In [9]:
[c[2].decode_text() if c is not None else None for c in result.history]

['<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n are you?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho you?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho are?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n you?<|im_end|>\n']

We can see that "?" was never removed, as it is present even in the empty mask. For rest, we can see that all system tokens are always present and only user tokens get masked. out between each calls.

Let's now analyze calculated shapley values.

In [10]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  Open, AI, ,,  desi...', shap_values=[nan, nan, ..., nan, nan])]]


Model was set to return just text tokens, so chat history has only text tokens. We can see that in the json representation inside ConversationEntry shap_values field is now populated. Nan values indicated that this token wasn't taken into calculation scope. As expected, we have 3 not-nan tokens. Let's see them.

In [11]:
user_entry = explained_chat_conversation[1][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,<|im_start|>,nan,2
1,user,nan,2
2,,nan,2
3,Who,0.390678,0
4,are,0.000000,0
5,you,0.609322,0
6,?,nan,0
7,<|im_end|>,nan,2
8,,nan,2


Let's create another turn to see how input significance will change:

In [12]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

And again, let's explain it:

In [13]:
result = explainer(
    chat=explained_chat, verbose=True, generation_kwargs=generation_kwargs
)

2026-05-05 13:32:01,389 - mllm_shap.shap.compact - INFO - Generating full response from the model...
2026-05-05 13:32:02,365 - mllm_shap.shap.base._masks_manager - INFO - Number of tokens for explainability: 6 (up to 63 additional calls)


Monte Carlo SHAP:   0%|          | 0/7 [00:00<?, ?it/s]

2026-05-05 13:32:06,447 - mllm_shap.shap.base._generate_responses - INFO - Generation stats: processed=7 cache_hits=0 cache_misses=7 skipped_filtered=0 model_elapsed_ms=3594.74
2026-05-05 13:32:06,448 - mllm_shap.shap.base.shap_explainer - INFO - Sampling stats: candidates=8 yielded=7 skipped(full_or_empty)=1 skipped(invalid)=0 skipped(duplicates)=0 elapsed_ms=3626.78


In [14]:
[c[2].decode_text() if c is not None else None for c in result.history]

['<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\n are you?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of inquiries and tasks. How can I help you today?<|im_end|><|im_end|>\n<|im_start|>user\nCan you repeat?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho you?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of inquiries and tasks. How can I help you today?<|im_end|><|im_end|>\n<|im_start|>user\nCan you repeat?<|im_end|>\n',
 '<|startoftext|><|im_start|>system\nYou are a helpful assistant that answers questions briefly.<|im_end|>\n<|im_start|>user\nWho are?<|im_end|>\n<|im_start|>assistant\nI am an AI language model created by OpenAI, designed to assist with a wide range of inqui

In [15]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, system, \n, You,  are,  a,  helpful,  assistant,  that,  answers,  questions,  briefly...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Who,  are,  you, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  Open, AI, ,,  desi...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, user, \n, Can,  you,  repeat, ?, <|im_end|>, \n...', shap_values=[nan, nan, ..., nan, nan])],
 [ChatEntry(content_type=0, roles=[SYSTEM, SYSTEM, ..., SYSTEM, SYSTEM], content='<|im_start|>, assistant, \n, I,  am,  an,  AI,  language,  model,  created,  by,  O

In [16]:
dt = []
for i in (1, 3):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,<|im_start|>,nan,2,1
1,user,nan,2,1
2,,nan,2,1
3,Who,0.250479,0,1
4,are,0.297093,0,1
5,you,0.237486,0,1
6,?,nan,0,1
7,<|im_end|>,nan,2,1
8,,nan,2,1
9,<|im_start|>,nan,2,3
